In [1]:
import os, glob, shutil, re
from typing import List, Tuple

# --- Utilidad: timestamp desde el nombre, ej. 1662344180.100000.png ---
_TS_RE = re.compile(r'(\d{10}(?:\.\d+)?)')  # 10 dígitos + opcional .xxxxx

def read_timestamp_from_name(path: str):
    m = _TS_RE.search(os.path.basename(path))
    return float(m.group(1)) if m else None

def _ensure_dir_empty(dst_dir: str):
    os.makedirs(dst_dir, exist_ok=True)
    if any(os.scandir(dst_dir)):
        raise RuntimeError(f"Destino '{dst_dir}' no está vacío. Elige otra carpeta o vacíala.")

def _link_or_copy(src: str, dst: str, mode: str = "hard"):
    """mode: 'hard' (por defecto), 'symlink', 'copy'"""
    if mode == "hard":
        try:
            os.link(src, dst)
        except Exception:
            shutil.copy2(src, dst)
    elif mode == "symlink":
        try:
            os.symlink(src, dst)
        except Exception:
            shutil.copy2(src, dst)
    elif mode == "copy":
        shutil.copy2(src, dst)
    else:
        raise ValueError("mode debe ser 'hard', 'symlink' o 'copy'.")

def _collect_frames(src_dir: str, exts=(".png",".jpg",".jpeg",".bmp")) -> List[str]:
    files = []
    for e in exts:
        files += glob.glob(os.path.join(src_dir, f"*{e}"))
    if not files:
        raise RuntimeError(f"No se encontraron imágenes en {src_dir}")
    return sorted(files)

# ------------------ MODO 1: cada N frames (stride) ------------------ #
def subsample_by_stride(src_dir: str, dst_dir: str, stride: int,
                        offset: int = 0, link_mode: str = "hard") -> Tuple[int,int]:
    """
    Copia/enlaza a dst_dir los frames [offset, offset+stride, ...]
    Retorna (total_origen, total_seleccionados)
    """
    if stride < 1:
        raise ValueError("stride debe ser >= 1")
    frames = _collect_frames(src_dir)
    _ensure_dir_empty(dst_dir)

    picked = frames[offset::stride]
    for src in picked:
        dst = os.path.join(dst_dir, os.path.basename(src))
        _link_or_copy(src, dst, mode=link_mode)

    print(f"[stride] origen={len(frames)}  seleccionados={len(picked)}  (stride={stride}, offset={offset})")
    return len(frames), len(picked)

# ------------- MODO 2: cada Δt segundos (con timestamps) ------------- #
def subsample_by_time(src_dir: str, dst_dir: str, dt_seconds: float,
                      tol: float = 0.11, start_at_first: bool = True,
                      link_mode: str = "hard") -> Tuple[int,int]:
    """
    Selecciona frames separados ~dt_seconds (± tol) según timestamp en el nombre.
    Empieza en el 1º frame (start_at_first=True) y luego busca el más cercano a t+Δt.
    Retorna (total_origen, total_seleccionados)
    """
    if dt_seconds <= 0:
        raise ValueError("dt_seconds debe ser > 0")
    frames = _collect_frames(src_dir)
    # construir lista con timestamps válidos
    pairs = []
    for p in frames:
        ts = read_timestamp_from_name(p)
        if ts is not None:
            pairs.append((ts, p))
    if not pairs:
        raise RuntimeError("No se encontraron timestamps en los nombres. Usa 'subsample_by_stride'.")
    pairs.sort(key=lambda x: x[0])  # por tiempo

    _ensure_dir_empty(dst_dir)

    picked = []
    if start_at_first:
        picked.append(pairs[0])
        next_target = pairs[0][0] + dt_seconds
        # recorrer y tomar el más cercano a cada múltiplo de dt
        i = 1
        while i < len(pairs):
            ts, path = pairs[i]
            # si ya superó (target - tol), considera ventana hasta (target + tol)
            if ts >= next_target - tol:
                # buscar mejor candidato en la ventana [target - tol, target + tol]
                best_j, best_err = i, abs(ts - next_target)
                j = i + 1
                while j < len(pairs) and pairs[j][0] <= next_target + tol:
                    err = abs(pairs[j][0] - next_target)
                    if err < best_err:
                        best_j, best_err = j, err
                    j += 1
                picked.append(pairs[best_j])
                next_target += dt_seconds
                i = best_j + 1
            else:
                i += 1
    else:
        # estrategia simple: *greedy* por salto mínimo >= dt_seconds - tol
        last_ts = pairs[0][0]
        for ts, path in pairs:
            if ts - last_ts >= (dt_seconds - tol):
                picked.append((ts, path))
                last_ts = ts

    # enlazar/copiar
    for _, src in picked:
        dst = os.path.join(dst_dir, os.path.basename(src))
        _link_or_copy(src, dst, mode=link_mode)

    print(f"[time] origen={len(frames)}  con_ts={len(pairs)}  seleccionados={len(picked)}  (Δt≈{dt_seconds}s, tol=±{tol}s)")
    return len(frames), len(picked)


In [ ]:
SRC = "carpeta_de_secuencia_completa"   # carpeta original de imágenes



# 2) Cada Δt segundos usando timestamps en el nombre (p. ej., 4 s y 8 s)
subsample_by_time(SRC, "carpeta_de_salida", dt_seconds=3.0, tol=0.11, link_mode="hard")



[time] origen=511  con_ts=511  seleccionados=18  (Δt≈3.0s, tol=±0.11s)


(511, 18)